In [14]:
import kagglehub
import shutil
from pathlib import Path

# Télécharger les datasets
path_jobs = kagglehub.dataset_download("ravindrasinghrana/job-description-dataset")
path_cv = kagglehub.dataset_download("suriyaganesh/resume-dataset-structured")

# Destinations
destination_job = Path("~/work/skill_match/data/job-description-dataset").expanduser()
destination_cv = Path("~/work/skill_match/data/resume-dataset-structured").expanduser()

# Copier les datasets
shutil.copytree(path_jobs, destination_job, dirs_exist_ok=True)
shutil.copytree(path_cv, destination_cv, dirs_exist_ok=True)

print("Job dataset copié dans :", destination_job)
print("CV dataset copié dans :", destination_cv)

Job dataset copié dans : /home/onyxia/work/skill_match/data/job-description-dataset
CV dataset copié dans : /home/onyxia/work/skill_match/data/resume-dataset-structured


In [1]:
import sys
sys.path.append("/home/onyxia/work/skill_match/skill_match")

# Prétraitement des jobs descriptions

In [3]:
import nltk
from nltk.corpus import stopwords

nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

[nltk_data] Downloading package stopwords to /home/onyxia/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [4]:
from preprocessing_job_description import clean_text, build_full_text, enable_tqdm_pandas

[nltk_data] Downloading package stopwords to /home/onyxia/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


## chargement du dataset job-description

In [6]:
import pandas as pd
jd = pd.read_csv("/home/onyxia/work/skill_match/data/job-description-dataset/job_descriptions.csv", on_bad_lines="skip")
print(jd.shape)
jd.head()

(1615940, 23)


,Job Id,Experience,Qualifications,Salary Range,location,Country,latitude,longitude,Work Type,Company Size,...,Contact,Job Title,Role,Job Portal,Job Description,Benefits,skills,Responsibilities,Company,Company Profile
0,1089843540111562,5 to 15 Years,M.Tech,$59K-$99K,Douglas,Isle of Man,54.2361,-4.5481,Intern,26801,...,001-381-930-7517x737,Digital Marketing Specialist,Social Media Manager,Snagajob,Social Media Managers oversee an organizations...,"{'Flexible Spending Accounts (FSAs), Relocatio...","Social media platforms (e.g., Facebook, Twitte...","Manage and grow social media accounts, create ...",Icahn Enterprises,"{""Sector"":""Diversified"",""Industry"":""Diversifie..."
1,398454096642776,2 to 12 Years,BCA,$56K-$116K,Ashgabat,Turkmenistan,38.9697,59.5563,Intern,100340,...,461-509-4216,Web Developer,Frontend Web Developer,Idealist,Frontend Web Developers design and implement u...,"{'Health Insurance, Retirement Plans, Paid Tim...","HTML, CSS, JavaScript Frontend frameworks (e.g...","Design and code user interfaces for websites, ...",PNC Financial Services Group,"{""Sector"":""Financial Services"",""Industry"":""Com..."
2,481640072963533,0 to 12 Years,PhD,$61K-$104K,Macao,"Macao SAR, China",22.1987,113.5439,Temporary,84525,...,9687619505,Operations Manager,Quality Control Manager,Jobs2Careers,Quality Control Managers establish and enforce...,"{'Legal Assistance, Bonuses and Incentive Prog...",Quality control processes and methodologies St...,Establish and enforce quality control standard...,United Services Automobile Assn.,"{""Sector"":""Insurance"",""Industry"":""Insurance: P..."
3,688192671473044,4 to 11 Years,PhD,$65K-$91K,Porto-Novo,Benin,9.3077,2.3158,Full-Time,129896,...,+1-820-643-5431x47576,Network Engineer,Wireless Network Engineer,FlexJobs,"Wireless Network Engineers design, implement, ...","{'Transportation Benefits, Professional Develo...",Wireless network design and architecture Wi-Fi...,"Design, configure, and optimize wireless netwo...",Hess,"{""Sector"":""Energy"",""Industry"":""Mining, Crude-O..."
4,117057806156508,1 to 12 Years,MBA,$64K-$87K,Santiago,Chile,-35.6751,-71.5429,Intern,53944,...,343.975.4702x9340,Event Manager,Conference Manager,Jobs2Careers,A Conference Manager coordinates and manages c...,"{'Flexible Spending Accounts (FSAs), Relocatio...",Event planning Conference logistics Budget man...,Specialize in conference and convention planni...,Cairn Energy,"{""Sector"":""Energy"",""Industry"":""Energy - Oil & ..."


## Construction et nettoyage du dataset

In [7]:
#barre de progression du suivi
enable_tqdm_pandas()
jd["raw_text"] = jd.progress_apply(build_full_text, axis=1)
jd["clean_text"] = jd["raw_text"].progress_apply(clean_text)


100%|██████████| 1615940/1615940 [04:12<00:00, 6411.32it/s]


In [ ]:
# vérification
jd[["Job Title", "Job Description", "skills", "Responsibilities", "clean_text"]].head()

In [8]:
# sauvegarde du dataset job-description
jd_clean = jd[["Role", "clean_text"]].copy()
output_path = "/home/onyxia/work/skill_match/data/jd_clean.csv"
jd_clean.to_csv(output_path, index=False)
print(f"Prétraitement terminé ! Fichier sauvegardé : {output_path}")

Prétraitement terminé ! Fichier sauvegardé : /home/onyxia/work/skill_match/data/jd_clean.csv


# Prétraitement CV

In [ ]:

from preprocessing_cv import ensure_list, build_cv

In [11]:
# le dataset est divisé en plusieurs parties , on charge celles dont on a besoin pour le nlp

# cv
people = pd.read_csv("/home/onyxia/work/skill_match/data/resume-dataset-structured/01_people.csv", dtype={"person_id": "int32"})
selected_ids = set(people["person_id"].tolist())

# abilities
abilities_dict = {}
for chunk in pd.read_csv("/home/onyxia/work/skill_match/data/resume-dataset-structured/02_abilities.csv", chunksize=200000, dtype={"person_id": "int32"}):
    chunk = chunk[chunk["person_id"].isin(selected_ids)]
    for pid, group in chunk.groupby("person_id")["ability"]:
        abilities_dict.setdefault(pid, []).extend(group.tolist())


# skills
skills_dict = {}
for chunk in pd.read_csv("/home/onyxia/work/skill_match/data/resume-dataset-structured/05_person_skills.csv", chunksize=200000, dtype={"person_id": "int32"}):
    chunk = chunk[chunk["person_id"].isin(selected_ids)]
    for pid, group in chunk.groupby("person_id")["skill"]:
        skills_dict.setdefault(pid, []).extend(group.tolist())

# education
edu = pd.read_csv("/home/onyxia/work/skill_match/data/resume-dataset-structured/03_education.csv", dtype={"person_id": "int32"})
edu = edu[edu["person_id"].isin(selected_ids)]
education_dict = {}
for pid, group in edu.groupby("person_id"):
    texts = []
    for _, row in group.iterrows():
        text = f"{row['program']} at {row['institution']} ({row['start_date']}) in {row['location']}"
        texts.append(text)
    education_dict[pid] = texts

# experiences
exp = pd.read_csv("/home/onyxia/work/skill_match/data/resume-dataset-structured/04_experience.csv", dtype={"person_id": "int32"})
exp = exp[exp["person_id"].isin(selected_ids)]
experience_dict = {}
for pid, group in exp.groupby("person_id"):
    texts = []
    for _, row in group.iterrows():
        text = f"{row['title']} at {row['firm']} ({row['start_date']} - {row['end_date']}) in {row['location']}"
        texts.append(text)
    experience_dict[pid] = texts


## Regroupement de toutes les parties dans un dataframe

In [12]:
people["abilities"] = people["person_id"].map(abilities_dict).apply(ensure_list)
people["skills"] = people["person_id"].map(skills_dict).apply(ensure_list)
people["education"] = people["person_id"].map(education_dict).apply(ensure_list)
people["experience"] = people["person_id"].map(experience_dict).apply(ensure_list)


### construction du texte du cv

In [13]:
people["cv_text"] = people.apply(build_cv, axis=1)

### Sauvegarde du fichier

In [14]:
output_path = "/home/onyxia/work/skill_match/data/cv_54k_reconstructed.csv"
people.to_csv(output_path, index=False)
print(f"Reconstruction terminée ! Fichier sauvegardé : {output_path}")

Reconstruction terminée ! Fichier sauvegardé : /home/onyxia/work/skill_match/data/cv_54k_reconstructed.csv


# NLP

In [15]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from tqdm import tqdm
import re
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

In [16]:
# chargement dataset
cv = pd.read_csv("/home/onyxia/work/skill_match/data/cv_54k_reconstructed.csv")
jd_clean = pd.read_csv("/home/onyxia/work/skill_match/data/jd_clean.csv")

In [32]:
## on va échantillonner les job-description par rapport aux catégories
TARGET_SIZE = 30000
role_counts = jd_clean["Role"].value_counts()
role_proportions = role_counts / role_counts.sum()
role_samples = (role_proportions * TARGET_SIZE).astype(int)

sampled_jd_list = []
for role, n_samples in role_samples.items():
    subset = jd_clean[jd_clean["Role"] == role]
    n_samples = min(n_samples, len(subset))
    sampled = subset.sample(n_samples, random_state=42)
    sampled_jd_list.append(sampled)

sampled_jd = pd.concat(sampled_jd_list).reset_index(drop=True)
jd = sampled_jd

In [33]:
# conversion en liste de textes
cv_texts = cv["cv_text"].fillna("").tolist()
jd_texts = jd["clean_text"].tolist()

## Vectorisation TF-IDF

In [22]:
from sklearn.feature_extraction.text import TfidfVectorizer

cv_texts = cv["cv_text"].fillna("").tolist()
jd_texts = jd["clean_text"].tolist()

all_texts = cv_texts + jd_texts

vectorizer = TfidfVectorizer(
    max_features=50000,
    stop_words="english"
)

tfidf_matrix = vectorizer.fit_transform(all_texts)

cv_tfidf = tfidf_matrix[:len(cv_texts)]
jd_tfidf = tfidf_matrix[len(cv_texts):]

## Récupération des top jd pour un cv

In [26]:
import importlib
import def_top_k

importlib.reload(def_top_k)

<module 'def_top_k' from '/home/onyxia/work/skill_match/skill_match/def_top_k.py'>

In [27]:
from def_top_k import top_k_tfidf, top_k_cv

example_cv = 0

matches = top_k_tfidf(example_cv, cv_tfidf, jd_tfidf, k=5)

for idx, score in matches:
    print(f"JD {idx} – score {score:.4f}")
    print(jd.loc[idx, "clean_text"][:200], "...\n")

JD 18919 – score 0.3104
database developer sql database developers design implement maintain relational databases using sql structured query language write queries optimize database performance ensure data integrity security ...

JD 18920 – score 0.3104
database developer sql database developers design implement maintain relational databases using sql structured query language write queries optimize database performance ensure data integrity security ...

JD 18876 – score 0.3104
database developer sql database developers design implement maintain relational databases using sql structured query language write queries optimize database performance ensure data integrity security ...

JD 18877 – score 0.3104
database developer sql database developers design implement maintain relational databases using sql structured query language write queries optimize database performance ensure data integrity security ...

JD 18864 – score 0.3104
database developer sql database developers design implem

## SBERT

In [34]:
# Importation du modèle
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2893.22it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [35]:
# embeddings (encodage textes)
cv_embeddings = model.encode(cv_texts, batch_size=64, show_progress_bar=True)
jd_embeddings = model.encode(jd_texts, batch_size=64, show_progress_bar=True)
np.save("cv_embeddings.npy", cv_embeddings)
np.save("jd_embeddings.npy", jd_embeddings)

Batches:   2%|▏         | 17/859 [13:52<11:27:09, 48.97s/it]


KeyboardInterrupt: 

In [ ]:
# chargement des embeddings
cv_embeddings = np.load("cv_embeddings.npy")
jd_embeddings = np.load("jd_embeddings.npy")

In [ ]:
example_cv = 0
matches = top_k_sbert(example_cv, cv_embeddings, jd_embeddings, k=5)

for idx, score in matches:
    print(f"JD {idx} – score {score:.4f}")
    print(jd.loc[idx, "clean_text"][:200], "...\n")

## NER

In [ ]:
from model_nlp import load_skill_ner, extract_skills, normalize_skills, compare_skills

nlp_skills = load_skill_ner()

In [ ]:
cv_skills_sets = cv["skills"].apply(normalize_skills).tolist()
jd_skills_lists = jd_clean["clean_text"].apply(lambda x: extract_skills(x, nlp_skills)).tolist()

In [ ]:
result = compare_skills(cv_idx=0, jd_idx=0, cv_skills_sets=cv_skills_sets, jd_skills_lists=jd_skills_lists)

print("Compétences présentes :", result["present"])
print("Compétences manquantes :", result["missing"])